In [5]:
from langchain_community.document_loaders import TextLoader

loader1 = TextLoader(file_path = "kb\\bio.txt",
                     encoding = "utf-8")
loader2 = TextLoader(file_path = "kb\\skills_and_projects.txt",
                     encoding = "utf-8")
loader3 = TextLoader(file_path = "kb\\career_goals.txt",
                     encoding = "utf-8")

doc1 = loader1.load()
doc2 = loader2.load()
doc3 = loader3.load()

documents = doc1 + doc2 + doc3
print(f'Total length doc {len(documents)}')


Total length doc 3


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)
chunks = splitter.split_documents(documents)
print(f'length of chunk {len(chunks)}')

length of chunk 12


In [9]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore

embeddings = HuggingFaceEmbeddings(model = "all-MiniLM-L6-v2")

vector_store = QdrantVectorStore.from_documents(
    documents = chunks,
    embedding = embeddings,
    url = "http://localhost:6333/",
    collection_name = "personal_assistant"
)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3835.28it/s]


In [ ]:
retriver = vector_store.as_retriever(search_kwards = {"k" : 5})



In [23]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [25]:
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_template("""
    Answer the question based only on the provided context
    If the answer is not in the context, say "I don't have enough information to answer that."
    context:
    {context}
                                              
    Question: {question}

    Answer:"""
)
print(f'{rag_prompt.input_variables}')

['context', 'question']


In [27]:
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
llm = ChatOllama(model = "qwen2.5:7b")
parser = StrOutputParser()

rag_chain = (
    {"context" : retriver | format_docs, "question" : RunnablePassthrough()} | rag_prompt | llm | parser  
)



In [38]:
query = "What is saim favourte food?"
for chunk in rag_chain.stream(query):
    print(chunk, end="", flush=True)

I don't have enough information to answer that.

In [13]:
response = llm.invoke("who are you")
print(response.content)

I am Qwen, an AI assistant created by Alibaba Cloud. I'm here to help with a wide range of tasks, from answering questions and providing information on various topics to assisting with writing and more. How can I assist you today?
